# Lab 1: Miniature Blockchain
**Course:** Blockchain & Application | **Instructor:** Vishnu Suresh | **Term:** Zima 2026/2027  
**Student:** Kaan Yazıcıoğlu | **File:** `yazicioglu_lab1.ipynb`

---
### Goal
Build small blockchain in Python. Test hashes, tamper detection, PoW mining, forks, and transaction checks.

---
## Part 1: Transactions and Hashes
Hash function take text, make 64-char hex fingerprint. Deterministic. One change flip whole hash (avalanche).

In [1]:
import hashlib
import json
import re
import time
from copy import deepcopy

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

print("Ready.")

Ready.


In [2]:
transactions = [
    "Kaan pays Vishnu 10",
    "Satoshi pays Vitalik 4",
    "Alice pays Bob 2",
]

for tx in transactions:
    print(f"{tx:<25} -> {sha256_text(tx)}")

Kaan pays Vishnu 10       -> fcf97621b046f6ce1bc8935acad2b619254e1c34c29f92a802d9df7de9d60325
Satoshi pays Vitalik 4    -> 157cea7961a5bfc6697b9c17848eae2550133df68ee3dcc3ef241a949f946e0e
Alice pays Bob 2          -> 5901bf0d768139978b3d164d208904666a819cd87c8a0ea864f553cf4bc1ff49


In [3]:
# Change 1 character: '10' -> '11'
t1 = "Kaan pays Vishnu 10"
t2 = "Kaan pays Vishnu 11"

print("Original:", sha256_text(t1))
print("Changed: ", sha256_text(t2))
print("Same length (64). Totally different hash. Avalanche effect.")

Original: fcf97621b046f6ce1bc8935acad2b619254e1c34c29f92a802d9df7de9d60325
Changed:  71e54be739c9befa2ff74709c4a6eab67590894372defdf6d0d747411bb43556
Same length (64). Totally different hash. Avalanche effect.


---
## Part 2: Linked Blocks
Block has: index, timestamp, tx list, previous_hash, nonce.
Hash computed over all fields except hash itself.

In [4]:
def calculate_block_hash(block: dict) -> str:
    data = {k: v for k, v in block.items() if k != "hash"}
    encoded = json.dumps(data, sort_keys=True, separators=(",", ":"))
    return sha256_text(encoded)

def make_block(index: int, transactions: list, previous_hash: str, timestamp: str, nonce: int = 0) -> dict:
    block = {
        "index": index,
        "timestamp": timestamp,
        "transactions": transactions,
        "previous_hash": previous_hash,
        "nonce": nonce,
    }
    block["hash"] = calculate_block_hash(block)
    return block

genesis = make_block(0, ["Course blockchain begins"], "0" * 64, "2026-10-01T09:00:00Z")
block_1 = make_block(1, ["Kaan pays Vishnu 10", "Satoshi pays Vitalik 4"], genesis["hash"], "2026-10-01T09:01:00Z")
block_2 = make_block(2, ["Alice pays Bob 2"], block_1["hash"], "2026-10-01T09:02:00Z")

chain = [genesis, block_1, block_2]

def show_chain(blocks: list) -> None:
    for b in blocks:
        print(f"Block {b['index']} | hash: {b['hash'][:12]}.. | prev: {b['previous_hash'][:12]}.. | nonce: {b['nonce']}")

show_chain(chain)

Block 0 | hash: 8311cd8d889a.. | prev: 000000000000.. | nonce: 0
Block 1 | hash: e5cd5f8d9aca.. | prev: 8311cd8d889a.. | nonce: 0
Block 2 | hash: 4d39f068422b.. | prev: e5cd5f8d9aca.. | nonce: 0


---
## Part 3: Tampering & Validation
Two rules:
1. Block hash == recalculated content hash.
2. Block previous_hash == prior block hash.

In [5]:
def validate_chain(blocks: list) -> tuple:
    for pos, b in enumerate(blocks):
        if b["hash"] != calculate_block_hash(b):
            return False, f"Block {pos} has changed contents"
        if pos > 0 and b["previous_hash"] != blocks[pos - 1]["hash"]:
            return False, f"Block {pos} has a broken previous-hash link"
    return True, "Chain is valid"

print("Original chain:", validate_chain(chain))

# Attack: change tx in Block 1
bad_chain = deepcopy(chain)
bad_chain[1]["transactions"][0] = "Kaan pays Vishnu 100"
print("After edit tx: ", validate_chain(bad_chain))

# Attack 2: recompute Block 1 hash to hide edit
bad_chain[1]["hash"] = calculate_block_hash(bad_chain[1])
print("After fix hash:", validate_chain(bad_chain))
print("-> Block 2 link breaks! Must rewrite whole chain to cheat.")

Original chain: (True, 'Chain is valid')
After edit tx:  (False, 'Block 1 has changed contents')
After fix hash: (False, 'Block 2 has a broken previous-hash link')
-> Block 2 link breaks! Must rewrite whole chain to cheat.


---
## Part 4: Simplified Proof of Work (PoW)
Keep changing nonce until hash starts with target zeros.
Hard to find nonce. Instant to check.

In [6]:
def mine_block(index: int, transactions: list, previous_hash: str, timestamp: str, difficulty: int = 3) -> tuple:
    block = {
        "index": index,
        "timestamp": timestamp,
        "transactions": transactions,
        "previous_hash": previous_hash,
        "nonce": 0,
    }
    target = "0" * difficulty
    attempts = 0
    while True:
        attempts += 1
        digest = calculate_block_hash(block)
        if digest.startswith(target):
            block["hash"] = digest
            return block, attempts
        block["nonce"] += 1

mined_block, attempts = mine_block(3, ["Vitalik pays Kaan 1"], chain[-1]["hash"], "2026-10-01T09:03:00Z", difficulty=3)
print(f"Mined! Attempts: {attempts} | Nonce: {mined_block['nonce']} | Hash: {mined_block['hash']}")

Mined! Attempts: 5107 | Nonce: 5106 | Hash: 0003ea59b1979b9db082824c5df5902516e69a463a28f879477b8f252ba6a3e1


In [7]:
# Compare difficulty: 2, 3, 4
print(f"{'Diff':<5} | {'Prefix':<7} | {'Attempts':<9} | {'Time (s)':<9} | {'Hash'}")
print("-" * 65)

for diff in [2, 3, 4]:
    t0 = time.perf_counter()
    b, atts = mine_block(3, ["Vitalik pays Kaan 1"], chain[-1]["hash"], "2026-10-01T09:03:00Z", difficulty=diff)
    dt = time.perf_counter() - t0
    print(f"{diff:<5} | {'0'*diff:<7} | {atts:<9} | {dt:<9.4f} | {b['hash'][:18]}..")

print("\nCaveman rule: each extra zero makes it ~16x harder.")

Diff  | Prefix  | Attempts  | Time (s)  | Hash
-----------------------------------------------------------------
2     | 00      | 388       | 0.0011    | 00f1bf0d23b2a92227..
3     | 000     | 5107      | 0.0154    | 0003ea59b1979b9db0..
4     | 0000    | 105827    | 0.2938    | 0000e7d44076c28523..

Caveman rule: each extra zero makes it ~16x harder.


---
## Part 5: Competing Histories (Forks)
Two miners find Block 3 at same time. Two valid chains.
Rule: longest chain wins.

In [8]:
branch_a = deepcopy(chain)
branch_b = deepcopy(chain)

block_a, _ = mine_block(3, ["Vitalik pays Kaan 1"], branch_a[-1]["hash"], "2026-10-01T09:03:00Z", difficulty=3)
block_b, _ = mine_block(3, ["Vitalik pays Satoshi 1"], branch_b[-1]["hash"], "2026-10-01T09:03:01Z", difficulty=3)

branch_a.append(block_a)
branch_b.append(block_b)

print("Branch A valid:", validate_chain(branch_a)[0])
print("Branch B valid:", validate_chain(branch_b)[0])

# Miner on branch A finds Block 4
block_a_2, _ = mine_block(4, ["Vishnu pays Alice 1"], branch_a[-1]["hash"], "2026-10-01T09:04:00Z", difficulty=3)
branch_a.append(block_a_2)

# Pick longest
preferred = max([branch_a, branch_b], key=len)
print(f"Preferred chain length: {len(preferred)} (Branch A won)")
show_chain(preferred)

Branch A valid: True
Branch B valid: True
Preferred chain length: 5 (Branch A won)
Block 0 | hash: 8311cd8d889a.. | prev: 000000000000.. | nonce: 0
Block 1 | hash: e5cd5f8d9aca.. | prev: 8311cd8d889a.. | nonce: 0
Block 2 | hash: 4d39f068422b.. | prev: e5cd5f8d9aca.. | nonce: 0
Block 3 | hash: 0003ea59b197.. | prev: 4d39f068422b.. | nonce: 5106
Block 4 | hash: 000462ad9600.. | prev: 0003ea59b197.. | nonce: 2875


---
## Assignments (Page 7)

### Task 1: Transaction Validation
Rules:
1. `Sender pays Receiver Amount`
2. Amount > 0
3. Sender != Receiver

In [9]:
TX_PATTERN = re.compile(r"^(\S+)\s+pays\s+(\S+)\s+(.+)$")

def validate_transaction(tx: str) -> tuple:
    if not isinstance(tx, str):
        return False, "Not string"
    match = TX_PATTERN.match(tx.strip())
    if not match:
        return False, "Bad format"
    sender, receiver, amt_str = match.groups()
    if sender == receiver:
        return False, "Self-pay forbidden"
    try:
        amt = float(amt_str)
    except ValueError:
        return False, "Amount not number"
    if amt <= 0:
        return False, "Amount must be > 0"
    return True, f"OK ({sender} -> {receiver}: {amt})"

tests = [
    "Kaan pays Vishnu 10",
    "Satoshi pays Vitalik 4.5",
    "Alice pays Bob 1",
    "Kaan pays Kaan 10",
    "Satoshi pays Vitalik -5",
    "Vishnu pays Bob 0",
    "Vitalik pays Alice abc",
    "Random garbage text",
]

for t in tests:
    ok, msg = validate_transaction(t)
    print(f"[{'VALID' if ok else 'FAIL'}] {t:<26} -> {msg}")

[VALID] Kaan pays Vishnu 10        -> OK (Kaan -> Vishnu: 10.0)
[VALID] Satoshi pays Vitalik 4.5   -> OK (Satoshi -> Vitalik: 4.5)
[VALID] Alice pays Bob 1           -> OK (Alice -> Bob: 1.0)
[FAIL] Kaan pays Kaan 10          -> Self-pay forbidden
[FAIL] Satoshi pays Vitalik -5    -> Amount must be > 0
[FAIL] Vishnu pays Bob 0          -> Amount must be > 0
[FAIL] Vitalik pays Alice abc     -> Amount not number
[FAIL] Random garbage text        -> Bad format


---
### Task 2: Block Integrity vs. Transaction Validity
- **Block Integrity:** Is the box locked and unbroken? (Hash match, prev_hash link, PoW).
- **Transaction Validity:** Is the thing inside the box legal? (Valid money, no self-pay, positive amount).

In [10]:
def validate_chain_full(blocks: list, check_tx: bool = True) -> tuple:
    for pos, b in enumerate(blocks):
        # 1. Block Integrity
        if b["hash"] != calculate_block_hash(b):
            return False, f"Block {pos}: bad hash"
        if pos > 0 and b["previous_hash"] != blocks[pos - 1]["hash"]:
            return False, f"Block {pos}: broken link"
        
        # 2. Transaction Validity
        if check_tx and pos > 0:
            for tx in b.get("transactions", []):
                ok, reason = validate_transaction(tx)
                if not ok:
                    return False, f"Block {pos} has invalid tx: '{tx}' ({reason})"
    return True, "All good"

# Make block with good hash & good link, but bad transactions
bad_tx_block, _ = mine_block(
    3,
    ["Kaan pays Kaan 100", "Satoshi pays Vitalik -50"],
    chain[-1]["hash"],
    "2026-10-01T09:05:00Z",
    difficulty=3
)
demo_chain = deepcopy(chain) + [bad_tx_block]

print("Test 1 - Check Block Integrity ONLY (check_tx=False):")
print("Result:", validate_chain_full(demo_chain, check_tx=False))
print("-> Block hash and link are 100% CORRECT.\n")

print("Test 2 - Check Full Chain WITH Transactions (check_tx=True):")
print("Result:", validate_chain_full(demo_chain, check_tx=True))
print("-> REJECTED! Block is sealed, but payload inside is illegal.")

Test 1 - Check Block Integrity ONLY (check_tx=False):
Result: (True, 'All good')
-> Block hash and link are 100% CORRECT.

Test 2 - Check Full Chain WITH Transactions (check_tx=True):
Result: (False, "Block 3 has invalid tx: 'Kaan pays Kaan 100' (Self-pay forbidden)")
-> REJECTED! Block is sealed, but payload inside is illegal.


---
### Caveman Summary: Block Integrity vs. Transaction Validity

| What | Block Integrity | Transaction Validity |
| :--- | :--- | :--- |
| **Meaning** | Box is unbroken | Money inside box is real |
| **Checked by** | SHA-256 hash, prev_hash, nonce | Tx rules (sender != receiver, amount > 0) |
| **Can fail alone?** | Yes, block can be good box holding bad money | Yes, bad tx can be sealed in good block |

**Real world (Bitcoin):** Miner must make good block AND good transactions. If single tx bad, whole block thrown away by network.